# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task215'
CH=10
H=W=30
FORBIDDEN={'Loop','Scan','NonZero','Unique','Script','Function'}
TASK_JSON=Path('/kaggle/input')/f'{TASK_ID}.json'

In [6]:
class VerticalTile3CanvasMask(nn.Module):
    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()
        fg = x[:,1:,:,:].sum(dim=1)
        row_fg = (fg.sum(dim=2) > 0.5).float()
        total = row_fg.sum(dim=1, keepdim=True)
        total_ok = (torch.abs(total - 3.0) < 0.5).float().view(1,1,1,1)
        acc = x * 0.0
        for top in range(0, H-2):
            valid = row_fg[:, top:top+1] * row_fg[:, top+1:top+2] * row_fg[:, top+2:top+3]
            valid = valid.view(1,1,1,1) * total_ok
            rows=[]
            for r in range(H):
                src = top + ((r - top) % 3)
                rows.append(x[:,:,src:src+1,:])
            tiled = torch.cat(rows, dim=2)
            acc = acc + valid * tiled
        return acc * active

model=VerticalTile3CanvasMask().eval()

In [7]:
def onehot_pad_zerooutside(grid, h=H, w=W, ch=CH):
    arr=np.zeros((1,ch,h,w), dtype=np.float32)
    gh,gw=len(grid),len(grid[0])
    for i in range(gh):
        for j in range(gw):
            arr[0,int(grid[i][j]),i,j]=1.0
    return arr

def pred_argmax(y):
    y=np.asarray(y)
    if y.ndim==4: y=y[0]
    return y.argmax(axis=0).astype(np.int64)

def validate_tensor_exact(sess, examples):
    ok=0
    for ex in examples:
        x=onehot_pad_zerooutside(ex['input'])
        y=sess.run(None, {'input':x})[0]
        target=onehot_pad_zerooutside(ex['output'])
        ok += int(np.array_equal(y, target))
    return ok, len(examples)

In [8]:
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
dummy[:,0,:,:]=1.0
onnx_path=Path(f'{TASK_ID}.onnx')
torch.onnx.export(model, dummy, onnx_path.as_posix(), input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False)
m=onnx.load(onnx_path.as_posix())
onnx.checker.check_model(m)
m=shape_inference.infer_shapes(m)
onnx.save(m, onnx_path.as_posix())
print('saved', onnx_path, 'bytes', onnx_path.stat().st_size)

/tmp/ipykernel_16/1326692398.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, onnx_path.as_posix(), input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False)


saved task215.onnx bytes 86130


In [9]:
m=onnx.load(f'{TASK_ID}.onnx')
ops=sorted(set(n.op_type for n in m.graph.node))
print('ops:', ops)
print('forbidden:', sorted(set(ops)&FORBIDDEN))
print('empty optional inputs:', sum(any(i=='' for i in n.input) for n in m.graph.node))
assert not (set(ops)&FORBIDDEN)
assert Path(f'{TASK_ID}.onnx').stat().st_size < 1_440_000
sess=ort.InferenceSession(f'{TASK_ID}.onnx', providers=['CPUExecutionProvider'])
if TASK_JSON.exists():
    D=json.load(open(TASK_JSON))
elif Path('/mnt/data/task215.json').exists():
    D=json.load(open('/mnt/data/task215.json'))
else:
    D=None
if D:
    print('train tensor exact:', validate_tensor_exact(sess,D['train']))
    print('visible test tensor exact:', validate_tensor_exact(sess,D['test']))
    print('arc-gen tensor exact:', validate_tensor_exact(sess,D.get('arc-gen',[])))
    assert validate_tensor_exact(sess,D['train'])[0] == len(D['train'])
    assert validate_tensor_exact(sess,D['test'])[0] == len(D['test'])

ops: ['Abs', 'Add', 'Cast', 'Concat', 'Constant', 'Greater', 'Less', 'Mul', 'ReduceSum', 'Reshape', 'Slice', 'Sub']
forbidden: []
empty optional inputs: 0


In [10]:
with zipfile.ZipFile('submission.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(f'{TASK_ID}.onnx', arcname=f'{TASK_ID}.onnx')
print('wrote submission.zip')

wrote submission.zip
